# 🧠 EEG Preprocessing — Left Hand vs Right Hand Classification

**Dataset:** EEGET-ALS Dataset (Healthy Participants Only)  
**Scenarios:** Scenario 1 (Lift Left Hand) & Scenario 2 (Lift Right Hand)  
**Tasks extracted:** `Thinking` (motor imagery) & `Acting` (physical movement)  

**Pipeline:**
1. Load encoding chaining CSV (precomputed binary features)
2. Filter: healthy subjects, scenarios 1 & 2, Thinking+Acting tasks only
3. Filter: target channels (C3, Cz, C4, FC3, FC4) & target subbands (Alpha, Beta, Gamma)
4. Engineer new features from chain_sequence
5. Pivot to wide format (1 row = 1 sample)
6. Save to `features_lh_rh.csv`

In [ ]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

print('Libraries loaded successfully.')

## 1. Configuration

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────────
ENCODING_CSV   = '/home/jeremy-mboe/Documents/Kuliah/Sem4/EEG_ALS/WEB/dataset/encoding chaining.csv'
OUTPUT_DIR     = '/home/jeremy-mboe/Documents/Kuliah/Sem4/EEG_ALS/WEB/classifier_notebook'
OUTPUT_CSV     = os.path.join(OUTPUT_DIR, 'features_lh_rh.csv')

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Target classes ─────────────────────────────────────────────────────────────
# scenario_id 1 = Lift Left Hand, 2 = Lift Right Hand
TARGET_SCENARIOS   = [1, 2]
LABEL_MAP          = {1: 'LH', 2: 'RH'}   # numeric label in output: 0=LH, 1=RH

# ── Tasks to keep (only motor-related) ─────────────────────────────────────────
# 'Thinking' = motor imagery (task i),  'Acting' = physical movement (task ii)
TARGET_TASKS = ['Thinking', 'Acting']

# ── Motor cortex channels (contralateral + ipsilateral) ────────────────────────
MOTOR_CHANNELS = ['C3', 'Cz', 'C4', 'FC3', 'FC4', 'CP3', 'CP4']

# ── Subbands most relevant to motor imagery ────────────────────────────────────
TARGET_SUBBANDS = ['Alpha', 'Beta', 'Gamma']

# ── Features in the CSV ────────────────────────────────────────────────────────
# Base features per (channel, subband): mav, variance, std, band_power,
# relative_power, peak_frequency  +  chain_sequence, chain_ratio

print('Configuration set.')
print(f'  Target scenarios : {TARGET_SCENARIOS}')
print(f'  Target tasks     : {TARGET_TASKS}')
print(f'  Motor channels   : {MOTOR_CHANNELS}')
print(f'  Target subbands  : {TARGET_SUBBANDS}')

## 2. Load Raw Encoding CSV

In [ ]:
print(f'Loading: {ENCODING_CSV}')
raw_df = pd.read_csv(ENCODING_CSV)

print(f'Shape  : {raw_df.shape}')
print(f'Columns: {list(raw_df.columns)}')
raw_df.head(3)

In [ ]:
# Quick inventory
print('=== Dataset Inventory ===')
print(f"Unique subjects    : {raw_df['subject_id'].nunique()}")
print(f"Unique scenarios   : {sorted(raw_df['scenario_id'].unique())}")
print(f"Unique tasks       : {raw_df['task'].unique()}")
print(f"Unique channels    : {sorted(raw_df['channel'].unique())}")
print(f"Unique subbands    : {raw_df['subband'].unique()}")
print(f"Unique features    : {raw_df['feature'].unique() if 'feature' in raw_df.columns else 'N/A'}")
print(f"chain_ratio range  : [{raw_df['chain_ratio'].min():.3f}, {raw_df['chain_ratio'].max():.3f}]")

## 3. Filter: Healthy Subjects + Target Scenarios + Target Tasks

In [ ]:
# Healthy subjects: subject_id starts with 'id' (not 'ALS')
mask_healthy  = raw_df['subject_id'].str.startswith('id')
mask_scenario = raw_df['scenario_id'].isin(TARGET_SCENARIOS)
mask_task     = raw_df['task'].isin(TARGET_TASKS)

filtered = raw_df[mask_healthy & mask_scenario & mask_task].copy()

print(f'After filter — shape: {filtered.shape}')
print(f"Subjects  : {filtered['subject_id'].nunique()}")
print(f"Scenarios : {filtered['scenario_id'].value_counts().to_dict()}")
print(f"Tasks     : {filtered['task'].value_counts().to_dict()}")

In [ ]:
# Further filter: motor channels & target subbands
available_channels = [c for c in MOTOR_CHANNELS if c in filtered['channel'].unique()]
available_subbands = [s for s in TARGET_SUBBANDS if s in filtered['subband'].unique()]

print(f'Available motor channels : {available_channels}')
print(f'Available target subbands: {available_subbands}')

filtered = filtered[
    filtered['channel'].isin(available_channels) &
    filtered['subband'].isin(available_subbands)
].copy()

print(f'After channel/subband filter — shape: {filtered.shape}')

## 4. Feature Engineering from chain_sequence

In [ ]:
def chain_features(chain_seq: str) -> dict:
    """
    Extract statistical features from binary chain_sequence string.
    chain_sequence encodes whether adjacent windows increased (1) or decreased (0).
    """
    if not isinstance(chain_seq, str) or len(chain_seq) == 0:
        return dict(chain_len=0, chain_ones=0, chain_zeros=0,
                    chain_ones_ratio=0.0, chain_longest_run1=0,
                    chain_longest_run0=0, chain_transitions=0,
                    chain_entropy=0.0)

    arr = np.array([int(c) for c in chain_seq])
    n   = len(arr)
    ones  = int(arr.sum())
    zeros = n - ones

    # Longest consecutive run of 1s
    def longest_run(bits, val):
        max_run = cur = 0
        for b in bits:
            if b == val:
                cur += 1
                max_run = max(max_run, cur)
            else:
                cur = 0
        return max_run

    # Transitions (0→1 or 1→0)
    transitions = int(np.sum(arr[1:] != arr[:-1]))

    # Binary entropy
    p1 = ones / n if n > 0 else 0.0
    p0 = 1.0 - p1
    eps = 1e-9
    entropy = -(p1 * np.log2(p1 + eps) + p0 * np.log2(p0 + eps))

    return dict(
        chain_len           = n,
        chain_ones          = ones,
        chain_zeros         = zeros,
        chain_ones_ratio    = ones / n,
        chain_longest_run1  = longest_run(arr, 1),
        chain_longest_run0  = longest_run(arr, 0),
        chain_transitions   = transitions,
        chain_entropy       = float(entropy),
    )


# Apply chain feature extraction
chain_feat_df = filtered['chain_sequence'].apply(chain_features).apply(pd.Series)
filtered = pd.concat([filtered.reset_index(drop=True), chain_feat_df], axis=1)

print('Chain features extracted:', list(chain_feat_df.columns))
filtered[list(chain_feat_df.columns)].describe().round(4)

## 5. Build Sample Key & Target Label

In [ ]:
# Each unique (subject_id, scenario_id, filename, task) = one sample
# label: 0 = LH (scenario 1), 1 = RH (scenario 2)
filtered['label']      = filtered['scenario_id'].map({1: 0, 2: 1})
filtered['label_name'] = filtered['scenario_id'].map(LABEL_MAP)

print('Label distribution:')
print(filtered[['label_name', 'label']].value_counts().to_string())

## 6. Determine Available Numeric Features

In [ ]:
# Numeric features per row (excluding meta columns)
META_COLS = {'subject_id', 'scenario', 'scenario_id', 'filename',
             'task', 'channel', 'subband', 'feature',
             'chain_sequence', 'label', 'label_name'}

NUMERIC_FEATS = [c for c in filtered.columns
                 if c not in META_COLS and pd.api.types.is_numeric_dtype(filtered[c])]

print(f'Numeric features per row ({len(NUMERIC_FEATS)}): {NUMERIC_FEATS}')

## 7. Pivot to Wide Format (1 row = 1 sample)

In [ ]:
# Group key: one sample = (subject_id, scenario_id, filename, task)
GROUP_KEY = ['subject_id', 'scenario_id', 'filename', 'task', 'label', 'label_name']

pivot_parts = []

for feat in NUMERIC_FEATS:
    # Create column name: channel_subband_feature
    tmp = filtered.copy()
    tmp['col_name'] = tmp['channel'] + '_' + tmp['subband'] + '_' + feat

    pivot = tmp.pivot_table(
        index=GROUP_KEY,
        columns='col_name',
        values=feat,
        aggfunc='mean'
    )
    pivot_parts.append(pivot)

wide_df = pd.concat(pivot_parts, axis=1)
wide_df = wide_df.reset_index()

# Remove duplicate columns if any
wide_df = wide_df.loc[:, ~wide_df.columns.duplicated()]

print(f'Wide format shape: {wide_df.shape}')
print(f'Samples per class:')
print(wide_df['label_name'].value_counts().to_string())
wide_df.head(3)

## 8. Handle Missing Values

In [ ]:
feature_cols = [c for c in wide_df.columns if c not in GROUP_KEY]

missing_pct = wide_df[feature_cols].isnull().mean()
print(f'Feature columns with >10% missing: {(missing_pct > 0.1).sum()}')

# Drop columns with >50% missing
to_drop = missing_pct[missing_pct > 0.5].index.tolist()
if to_drop:
    print(f'Dropping {len(to_drop)} columns (>50% NaN): {to_drop[:5]} ...')
    wide_df = wide_df.drop(columns=to_drop)
    feature_cols = [c for c in wide_df.columns if c not in GROUP_KEY]

# Fill remaining NaN with column median
wide_df[feature_cols] = wide_df[feature_cols].fillna(wide_df[feature_cols].median())
print(f'Remaining NaN after fill: {wide_df[feature_cols].isnull().sum().sum()}')

## 9. Missing Value Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
sample_feats = feature_cols[:60]  # show first 60 for readability
miss_matrix  = wide_df[sample_feats].isnull().astype(int)
sns.heatmap(miss_matrix, cbar=False, ax=ax,
            cmap='Reds', yticklabels=False)
ax.set_title('Missing Values Heatmap (first 60 features, red = missing)')
ax.set_xlabel('Feature')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'missing_heatmap.png'), dpi=100)
plt.show()
print('Plot saved.')

## 10. Class Balance Check

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Overall balance
counts = wide_df['label_name'].value_counts()
axes[0].bar(counts.index, counts.values, color=['steelblue', 'coral'], edgecolor='black')
axes[0].set_title('Overall Class Distribution')
axes[0].set_ylabel('Number of Samples')
for i, (name, v) in enumerate(counts.items()):
    axes[0].text(i, v + 1, str(v), ha='center', fontweight='bold')

# Per task
task_counts = wide_df.groupby(['task', 'label_name']).size().unstack(fill_value=0)
task_counts.plot(kind='bar', ax=axes[1], color=['steelblue', 'coral'], edgecolor='black')
axes[1].set_title('Class Distribution per Task')
axes[1].set_xlabel('Task')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=0)
axes[1].legend(title='Class')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'class_balance.png'), dpi=100)
plt.show()

## 11. Feature Statistics Summary

In [ ]:
desc = wide_df[feature_cols].describe().T
desc['cv'] = desc['std'] / (desc['mean'].abs() + 1e-9)  # coefficient of variation
print(f'Total features: {len(feature_cols)}')
print('\nTop 10 highest-variance features:')
desc.sort_values('std', ascending=False).head(10)[['mean','std','min','max','cv']]

## 12. Correlation with Label

In [ ]:
from scipy.stats import pointbiserialr

corr_scores = {}
for col in feature_cols:
    vals = wide_df[col].values
    if np.std(vals) > 0:
        r, p = pointbiserialr(wide_df['label'].values, vals)
        corr_scores[col] = (abs(r), p)

corr_df = pd.DataFrame(corr_scores, index=['abs_corr', 'pvalue']).T
corr_df = corr_df.sort_values('abs_corr', ascending=False)

print('Top 15 features by |correlation| with label:')
print(corr_df.head(15).round(4).to_string())

# Plot top 20
fig, ax = plt.subplots(figsize=(12, 5))
top20 = corr_df.head(20)
ax.barh(top20.index[::-1], top20['abs_corr'].values[::-1], color='steelblue')
ax.set_xlabel('|Point-Biserial Correlation|')
ax.set_title('Top 20 Features by Correlation with LH/RH Label')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'feature_correlation.png'), dpi=100)
plt.show()

## 13. Save Features CSV

In [ ]:
wide_df.to_csv(OUTPUT_CSV, index=False)

print(f'✅ Features saved to: {OUTPUT_CSV}')
print(f'   Shape : {wide_df.shape}')
print(f'   Rows  : {len(wide_df)}  (samples)')
print(f'   Cols  : {len(wide_df.columns)}  ({len(feature_cols)} features + {len(GROUP_KEY)} meta)')
print(f'   LH    : {(wide_df["label"]==0).sum()}')
print(f'   RH    : {(wide_df["label"]==1).sum()}')

# Save summary
summary = pd.DataFrame({
    'metric': ['n_samples','n_features','n_LH','n_RH','n_subjects',
               'n_tasks','channels_used','subbands_used'],
    'value': [
        len(wide_df), len(feature_cols),
        int((wide_df['label']==0).sum()),
        int((wide_df['label']==1).sum()),
        wide_df['subject_id'].nunique(),
        wide_df['task'].nunique(),
        str(available_channels),
        str(available_subbands)
    ]
})
summary.to_csv(os.path.join(OUTPUT_DIR, 'preprocessing_summary.csv'), index=False)
print('\nPreprocessing summary saved.')
summary